In [344]:
print("Hello, World!")
#https://drive.google.com/file/d/1ByM_1y4W26dqik6hj65f_brQklHoDr0W/view

Hello, World!


In [345]:
import numpy as np
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [346]:
#!pip install xgboost


In [441]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
import xgboost as xgb


In [348]:
import matplotlib.pyplot as plt
import seaborn as sns

In [349]:
import joblib
import pickle

In [350]:
# Load datasets silently
fact_appt = pd.read_csv('fact_appointments.csv')
dim_patients = pd.read_csv('dim_patients.csv')
dim_providers = pd.read_csv('dim_providers.csv')
dim_clinics = pd.read_csv('dim_clinics.csv')
dim_dates = pd.read_csv('dim_dates.csv')


In [351]:
fact_appt = pd.read_csv('fact_appointments.csv')
print(fact_appt)

       APPOINTMENT_ID PATIENT_ID CLINIC_ID PROVIDER_ID  DATE_ID  \
0            A0000001    P004264      C007     PR00120        1   
1            A0000002    P002201      C002     PR00045        2   
2            A0000003    P002984      C012     PR00105        3   
3            A0000004    P002734      C012     PR00085        4   
4            A0000005    P002814      C015     PR00189        5   
...               ...        ...       ...         ...      ...   
119995       A0119996    P000603      C015     PR00111   136016   
119996       A0119997    P000153      C004     PR00202   136017   
119997       A0119998    P004182      C004     PR00058   136018   
119998       A0119999    P001633      C005     PR00030   136019   
119999       A0120000    P002228      C015     PR00019   136020   

       APPOINTMENT_TIME APPOINTMENT_DATE  LEAD_TIME_DAYS  WAIT_TIME_MINUTES  \
0            9:00:00 AM       2023-06-11               5               21.0   
1            3:30:00 PM       2023-05

In [352]:
fact_appt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   APPOINTMENT_ID     120000 non-null  object 
 1   PATIENT_ID         120000 non-null  object 
 2   CLINIC_ID          120000 non-null  object 
 3   PROVIDER_ID        120000 non-null  object 
 4   DATE_ID            120000 non-null  int64  
 5   APPOINTMENT_TIME   120000 non-null  object 
 6   APPOINTMENT_DATE   120000 non-null  object 
 7   LEAD_TIME_DAYS     120000 non-null  int64  
 8   WAIT_TIME_MINUTES  98656 non-null   float64
 9   IS_NO_SHOW         120000 non-null  int64  
dtypes: float64(1), int64(3), object(6)
memory usage: 9.2+ MB


In [353]:
fact_appt.shape

(120000, 10)

In [354]:
print(dim_patients)

     PATIENT_ID INSURANCE_TYPE  AGE
0       P004264         Public   67
1       P002984       Self-pay   51
2       P000389        Private   12
3       P003568        Private   42
4       P004069         Public   76
...         ...            ...  ...
4995    P001738         Public   22
4996    P002078       Self-pay   20
4997    P000542        Private   47
4998    P000019         Public   68
4999    P000004        Private   57

[5000 rows x 3 columns]


In [355]:
print(dim_providers)

    PROVIDER_ID       SPECIALTY CLINIC_ID CLINIC_ASSIGNMENT
0       PR00017     Orthopedics      C003         Full-time
1       PR00098        Oncology      C007         Full-time
2       PR00052     Orthopedics      C003         Full-time
3       PR00133     Dermatology      C005         Part-time
4       PR00003      Psychiatry      C012         Full-time
..          ...             ...       ...               ...
215     PR00140      Psychiatry      C012         Full-time
216     PR00190      Psychiatry      C012         Full-time
217     PR00068  Otolaryngology      C008         Part-time
218     PR00095      Cardiology      C002         Full-time
219     PR00146     Pulmonology      C015         Full-time

[220 rows x 4 columns]


In [356]:
dim_providers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220 entries, 0 to 219
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   PROVIDER_ID        220 non-null    object
 1   SPECIALTY          220 non-null    object
 2   CLINIC_ID          220 non-null    object
 3   CLINIC_ASSIGNMENT  220 non-null    object
dtypes: object(4)
memory usage: 7.0+ KB


In [357]:
dim_providers.shape

(220, 4)

In [358]:
print(dim_clinics.head())

  appointment_id patient_id provider_id appointment_date appointment_time  \
0       A0000001    P004264     PR00120        6/11/2023       9:00:00 AM   
1       A0000002    P002201     PR00045         5/3/2023       3:30:00 PM   
2       A0000003    P002984     PR00105         7/9/2024       4:45:00 PM   
3       A0000004    P002734     PR00085         7/7/2024       3:45:00 PM   
4       A0000005    P002814     PR00189        7/17/2024       5:15:00 PM   

   lead_time_days  wait_time_minutes  is_no_show_0_1  age insurance_type  \
0               5               21.0               0   67         Public   
1               4               27.0               0   12        Private   
2               7               19.0               0   51       Self-pay   
3              17               23.0               0   31        Private   
4              24               23.0               0    3        Private   

     specialty provider_clinic_id clinic_assignment         clinic_name  \
0    

In [359]:
dim_clinics.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 19 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   appointment_id       120000 non-null  object 
 1   patient_id           120000 non-null  object 
 2   provider_id          120000 non-null  object 
 3   appointment_date     120000 non-null  object 
 4   appointment_time     120000 non-null  object 
 5   lead_time_days       120000 non-null  int64  
 6   wait_time_minutes    98656 non-null   float64
 7   is_no_show_0_1       120000 non-null  int64  
 8   age                  120000 non-null  int64  
 9   insurance_type       120000 non-null  object 
 10  specialty            120000 non-null  object 
 11  provider_clinic_id   120000 non-null  object 
 12  clinic_assignment    120000 non-null  object 
 13  clinic_name          120000 non-null  object 
 14  city                 120000 non-null  object 
 15  hours            

In [360]:
dim_clinics.shape

(120000, 19)

In [361]:
print(dim_dates)

      DATE_ID DAY_OF_WEEK APPOINTMENT_DATE MONTH_NAME  YEAR
0           1         Sun       2023-06-11        Jun  2023
1           2         Thu       2024-07-04        Jul  2024
2           3         Wed       2025-09-17        Sep  2025
3           4         Thu       2025-03-20        Mar  2025
4           5         Thu       2025-05-01        May  2025
...       ...         ...              ...        ...   ...
1091     1445         Sun       2023-11-19        Nov  2023
1092     1446         Sun       2024-10-20        Oct  2024
1093     1447         Thu       2024-01-11        Jan  2024
1094     1448         Wed       2025-10-01        Oct  2025
1095     1449         Wed       2024-12-18        Dec  2024

[1096 rows x 5 columns]


In [362]:
dim_dates.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1096 entries, 0 to 1095
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   DATE_ID           1096 non-null   int64 
 1   DAY_OF_WEEK       1096 non-null   object
 2   APPOINTMENT_DATE  1096 non-null   object
 3   MONTH_NAME        1096 non-null   object
 4   YEAR              1096 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 42.9+ KB


In [363]:
dim_dates.shape

(1096, 5)

In [364]:
# =============================================================================
# 1. LOAD ALL DATASETS
# =============================================================================

print("\n[1] Loading datasets...")

# Load fact table (main appointments)
fact_appt = pd.read_csv('fact_appointments.csv')
print(f"  - Fact appointments: {len(fact_appt)} rows")

# Load dimension tables
dim_patients = pd.read_csv('dim_patients.csv')
print(f"  - Dim patients: {len(dim_patients)} rows")

dim_providers = pd.read_csv('dim_providers.csv')
print(f"  - Dim providers: {len(dim_providers)} rows")

dim_clinics = pd.read_csv('dim_clinics.csv')  # Note: filename has typo 'clinis'
print(f"  - Dim clinics: {len(dim_clinics)} rows")

dim_dates = pd.read_csv('dim_dates.csv')
print(f"  - Dim dates: {len(dim_dates)} rows")



[1] Loading datasets...
  - Fact appointments: 120000 rows
  - Dim patients: 5000 rows
  - Dim providers: 220 rows
  - Dim clinics: 120000 rows
  - Dim dates: 1096 rows


In [365]:
# =============================================================================
# 2. STANDARDIZE COLUMN NAMES
# =============================================================================

print("\n[2] Standardizing column names...")

# Rename columns to match across datasets
dim_clinics = dim_clinics.rename(columns={
    'provider_clinic_id': 'CLINIC_ID', 
    'clinic_name': 'CLINIC_NAME',
    'city': 'CITY',
    'hours': 'HOURS'
})

# Convert all column names to uppercase
dim_clinics.columns = dim_clinics.columns.str.upper()

# Dim providers has CLINIC_ID already, good!
# Dim patients has PATIENT_ID, INSURANCE_TYPE, AGE
# Dim data has DATE_ID, DATE, DAY_Of_WEEK


[2] Standardizing column names...


In [366]:
dim_clinics['CLINIC_ID'].value_counts().head(10)
dim_clinics[dim_clinics['CLINIC_ID'].duplicated(keep=False)]
dim_clinics = dim_clinics.drop_duplicates(subset='CLINIC_ID')
dim_clinics = dim_clinics.sort_values('CLINIC_ID').drop_duplicates('CLINIC_ID')


In [369]:
#=============================================================================
# 3. MERGE STEP BY STEP
# =============================================================================

print("\n[3] Merging datasets...")

# Start with fact table
merged = fact_appt.copy()
print(f"  - Started with fact table: {len(merged)} rows")

# Merge with patients
merged = merged.merge(
    dim_patients[['PATIENT_ID', 'INSURANCE_TYPE', 'AGE']], 
    on='PATIENT_ID', 
    how='left'
)
print(f"  - After merging patients: {len(merged)} rows")

# Merge with providers (to get specialty)
merged = merged.merge(
    dim_providers[['PROVIDER_ID', 'SPECIALTY']], 
    on='PROVIDER_ID', 
    how='left'
)
print(f"  - After merging providers: {len(merged)} rows")

# Merge with clinics (to get clinic details)
merged['CLINIC_ID'].value_counts().head()
dim_clinics['CLINIC_ID'].value_counts().head()

# Merge with clinics (to get clinic details)
merged = merged.merge(
    dim_clinics[['CLINIC_ID', 'CLINIC_NAME', 'CITY', 'HOURS', 'CLINIC_ASSIGNMENT']],
    on='CLINIC_ID',
    how='left',
)

print(f"  - After merging clinics: {len(merged)} rows")

print(f"  - Merging fact_appt, dim_patients and dim_providers: {len(merged)} rows")
print(merged.head())



[3] Merging datasets...
  - Started with fact table: 120000 rows
  - After merging patients: 120000 rows
  - After merging providers: 120000 rows
  - After merging clinics: 120000 rows
  - Merging fact_appt, dim_patients and dim_providers: 120000 rows
  APPOINTMENT_ID PATIENT_ID CLINIC_ID PROVIDER_ID  DATE_ID APPOINTMENT_TIME  \
0       A0000001    P004264      C007     PR00120        1       9:00:00 AM   
1       A0000002    P002201      C002     PR00045        2       3:30:00 PM   
2       A0000003    P002984      C012     PR00105        3       4:45:00 PM   
3       A0000004    P002734      C012     PR00085        4       3:45:00 PM   
4       A0000005    P002814      C015     PR00189        5       5:15:00 PM   

  APPOINTMENT_DATE  LEAD_TIME_DAYS  WAIT_TIME_MINUTES  IS_NO_SHOW  \
0       2023-06-11               5               21.0           0   
1       2023-05-03               4               27.0           0   
2       2024-07-09               7               19.0           0

In [371]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 17 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   APPOINTMENT_ID     120000 non-null  object 
 1   PATIENT_ID         120000 non-null  object 
 2   CLINIC_ID          120000 non-null  object 
 3   PROVIDER_ID        120000 non-null  object 
 4   DATE_ID            120000 non-null  int64  
 5   APPOINTMENT_TIME   120000 non-null  object 
 6   APPOINTMENT_DATE   120000 non-null  object 
 7   LEAD_TIME_DAYS     120000 non-null  int64  
 8   WAIT_TIME_MINUTES  98656 non-null   float64
 9   IS_NO_SHOW         120000 non-null  int64  
 10  INSURANCE_TYPE     120000 non-null  object 
 11  AGE                120000 non-null  int64  
 12  SPECIALTY          120000 non-null  object 
 13  CLINIC_NAME        120000 non-null  object 
 14  CITY               120000 non-null  object 
 15  HOURS              120000 non-null  object 
 16  CL

In [408]:
# =============================================================================
# 4. CREATE FINAL DATASET IN THE FORMAT WE NEED
# =============================================================================

print("\n[4] Creating final dataset format...")

# Map DATE_ID to actual date if needed (simplified - using APPOINTMENT_DATE from fact)
# The dim_dates table would have more date attributes, but we'll use what we have

# Create final dataframe with all columns in our original format
final_df = merged[[
    'APPOINTMENT_ID', 'PATIENT_ID', 'PROVIDER_ID', 
    'APPOINTMENT_DATE', 'APPOINTMENT_TIME', 
    'LEAD_TIME_DAYS', 'WAIT_TIME_MINUTES', 'IS_NO_SHOW',
    'AGE', 'INSURANCE_TYPE', 'SPECIALTY',
    'CLINIC_ID', 'CLINIC_ASSIGNMENT', 'CLINIC_NAME', 'CITY', 'HOURS'
]].copy()

# Rename columns to match our original working file
final_df = final_df.rename(columns={
    'APPOINTMENT_ID': 'appointment_id',
    'PATIENT_ID': 'patient_id',
    'PROVIDER_ID': 'provider_id',
    'APPOINTMENT_DATE': 'appointment_date',
    'APPOINTMENT_TIME': 'appointment_time',
    'LEAD_TIME_DAYS': 'lead_time_days',
    'WAIT_TIME_MINUTES': 'wait_time_minutes',
    'IS_NO_SHOW': 'is_no_show_0_1',
    'AGE': 'age',
    'INSURANCE_TYPE': 'insurance_type',
    'SPECIALTY': 'specialty',
    'CLINIC_ID': 'provider_clinic_id',
    'CLINIC_ASSIGNMENT': 'clinic_assignment',
    'CLINIC_NAME': 'clinic_name',
    'CITY': 'city',
    'HOURS': 'hours'
})

# Add the metadata columns (these were in original)
final_df['ingestion_timestamp'] = pd.Timestamp.now().timestamp() * 1000
final_df['source_system'] = 'merged_dataset'
final_df['batch_id'] = 'merge_' + pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')


[4] Creating final dataset format...


In [409]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 19 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   appointment_id       120000 non-null  object 
 1   patient_id           120000 non-null  object 
 2   provider_id          120000 non-null  object 
 3   appointment_date     120000 non-null  object 
 4   appointment_time     120000 non-null  object 
 5   lead_time_days       120000 non-null  int64  
 6   wait_time_minutes    98656 non-null   float64
 7   is_no_show_0_1       120000 non-null  int64  
 8   age                  120000 non-null  int64  
 9   insurance_type       120000 non-null  object 
 10  specialty            120000 non-null  object 
 11  provider_clinic_id   120000 non-null  object 
 12  clinic_assignment    120000 non-null  object 
 13  clinic_name          120000 non-null  object 
 14  city                 120000 non-null  object 
 15  hours            

In [410]:
# Clean whitespace
final_df['appointment_date'] = pd.to_datetime(final_df['appointment_date'].astype(str).str.strip())

# Convert to datetime safely
final_df['appointment_time'] = pd.to_datetime(
    final_df['appointment_time'], 
    format='%I:%M:%S %p').dt.time



In [411]:
final_df['appointment_date'] = (
    final_df['appointment_date'] +
    pd.to_timedelta(final_df['appointment_time'].astype(str))
)


In [412]:
final_df['appointment_date'].head()

0   2023-06-11 09:00:00
1   2023-05-03 15:30:00
2   2024-07-09 16:45:00
3   2024-07-07 15:45:00
4   2024-07-17 17:15:00
Name: appointment_date, dtype: datetime64[ns]

In [413]:
final_df['appointment_year'] = final_df['appointment_date'].dt.year
final_df['appointment_month'] = final_df['appointment_date'].dt.month
final_df['appointment_day'] = final_df['appointment_date'].dt.day
final_df['appointment_dayofweek'] = final_df['appointment_date'].dt.dayofweek #0-monday, 1- tuesday
final_df['appointment_quarter'] = final_df['appointment_date'].dt.quarter

In [414]:
final_df['appointment_hour'] = pd.to_datetime(final_df['appointment_time'].astype(str)).dt.hour

In [415]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 25 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   appointment_id         120000 non-null  object        
 1   patient_id             120000 non-null  object        
 2   provider_id            120000 non-null  object        
 3   appointment_date       120000 non-null  datetime64[ns]
 4   appointment_time       120000 non-null  object        
 5   lead_time_days         120000 non-null  int64         
 6   wait_time_minutes      98656 non-null   float64       
 7   is_no_show_0_1         120000 non-null  int64         
 8   age                    120000 non-null  int64         
 9   insurance_type         120000 non-null  object        
 10  specialty              120000 non-null  object        
 11  provider_clinic_id     120000 non-null  object        
 12  clinic_assignment      120000 non-null  obje

In [402]:
final_df.shape

(120000, 25)

In [418]:
final_df.isnull().sum()

appointment_id           0
patient_id               0
provider_id              0
appointment_date         0
appointment_time         0
lead_time_days           0
wait_time_minutes        0
is_no_show_0_1           0
age                      0
insurance_type           0
specialty                0
provider_clinic_id       0
clinic_assignment        0
clinic_name              0
city                     0
hours                    0
ingestion_timestamp      0
source_system            0
batch_id                 0
appointment_year         0
appointment_month        0
appointment_day          0
appointment_dayofweek    0
appointment_quarter      0
appointment_hour         0
dtype: int64

In [419]:
# =============================================================================
# 5. CHECK FOR MISSING VALUES
# =============================================================================

print("\n[5] Checking data quality...")

missing_counts = final_df.isnull().sum()
missing_cols = missing_counts[missing_counts > 0]

if len(missing_cols) > 0:
    print("  ⚠️ Missing values found:")
    for col, count in missing_cols.items():
        print(f"    - {col}: {count} missing ({count/len(final_df)*100:.1f}%)")
else:
    print("  ✅ No missing values!")

# Handle missing values (especially wait_time_minutes)
print("\n[6] Handling missing values...")

# Fill missing wait_time_minutes with median by specialty
if final_df['wait_time_minutes'].isnull().any():
    # Calculate median per specialty
    specialty_medians = final_df.groupby('specialty')['wait_time_minutes'].median()
    
    # Fill missing values
    for specialty, median_wait in specialty_medians.items():
        mask = (final_df['specialty'] == specialty) & (final_df['wait_time_minutes'].isnull())
        final_df.loc[mask, 'wait_time_minutes'] = median_wait
    
    # If still missing, use global median
    global_median = final_df['wait_time_minutes'].median()
    final_df['wait_time_minutes'].fillna(global_median, inplace=True)
    final_df['wait_time_minutes'] = final_df['wait_time_minutes'].fillna(global_median)

    
    print(f"  ✅ Filled missing wait times (used specialty medians)")

# Fill any other missing values
final_df.fillna({
    'insurance_type': 'Unknown',
    'specialty': 'Unknown',
    'clinic_assignment': 'Unknown'
}, inplace=True)


[5] Checking data quality...
  ✅ No missing values!

[6] Handling missing values...


In [420]:
# =============================================================================
# 6. SAVE FINAL DATASET
# =============================================================================

print("\n[7] Saving final dataset...")

output_file = 'healthpulse_db_merged.csv'
final_df.to_csv(output_file, index=False)

print(f"\n✅ MERGE COMPLETE!")
print(f"   Final dataset: {len(final_df)} rows, {len(final_df.columns)} columns")
print(f"   Saved to: {output_file}")

# Show sample
print("\n[Sample of merged data (first 3 rows)]:")
print(final_df[['appointment_id', 'patient_id', 'age', 'insurance_type', 
                'specialty', 'wait_time_minutes', 'is_no_show_0_1']].head(3))

# Show statistics
print("\n[Dataset Statistics]:")
print(f"  - Total appointments: {len(final_df)}")
print(f"  - No-show rate: {final_df['is_no_show_0_1'].mean()*100:.1f}%")
print(f"  - Unique patients: {final_df['patient_id'].nunique()}")
print(f"  - Unique providers: {final_df['provider_id'].nunique()}")
print(f"  - Clinics: {final_df['clinic_name'].nunique()}")
print(f"  - Specialties: {final_df['specialty'].nunique()}")
print(f"  - Avg lead time: {final_df['lead_time_days'].mean():.1f} days")
print(f"  - Avg wait time: {final_df['wait_time_minutes'].mean():.1f} minutes")

print("\n" + "="*60)


[7] Saving final dataset...

✅ MERGE COMPLETE!
   Final dataset: 120000 rows, 25 columns
   Saved to: healthpulse_db_merged.csv

[Sample of merged data (first 3 rows)]:
  appointment_id patient_id  age insurance_type   specialty  \
0       A0000001    P004264   67         Public    Oncology   
1       A0000002    P002201   12        Private  Cardiology   
2       A0000003    P002984   51       Self-pay  Psychiatry   

   wait_time_minutes  is_no_show_0_1  
0               21.0               0  
1               27.0               0  
2               19.0               0  

[Dataset Statistics]:
  - Total appointments: 120000
  - No-show rate: 17.8%
  - Unique patients: 5000
  - Unique providers: 220
  - Clinics: 15
  - Specialties: 15
  - Avg lead time: 14.0 days
  - Avg wait time: 21.2 minutes



In [421]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 25 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   appointment_id         120000 non-null  object        
 1   patient_id             120000 non-null  object        
 2   provider_id            120000 non-null  object        
 3   appointment_date       120000 non-null  datetime64[ns]
 4   appointment_time       120000 non-null  object        
 5   lead_time_days         120000 non-null  int64         
 6   wait_time_minutes      120000 non-null  float64       
 7   is_no_show_0_1         120000 non-null  int64         
 8   age                    120000 non-null  int64         
 9   insurance_type         120000 non-null  object        
 10  specialty              120000 non-null  object        
 11  provider_clinic_id     120000 non-null  object        
 12  clinic_assignment      120000 non-null  obje

In [422]:
#=============================================================================
# 7. ENGINEER FEATURES
# =============================================================================
def engineer_features(final_df):
    ''"Create advanced features for no-show prediction"""
    
    df_feat = final_df.copy()
    
    # 1. Time-based features
    df_feat['is_monday'] = (df_feat['appointment_dayofweek'] == 0).astype(int)
    df_feat['is_friday'] = (df_feat['appointment_dayofweek'] == 4).astype(int)
    df_feat['is_weekend'] = (df_feat['appointment_dayofweek'] >= 5).astype(int)   
    
    # 2. Hour-based features
    df_feat['is_morning'] = ((df_feat['appointment_hour'] >= 8) &
                             (df_feat['appointment_hour'] < 12)).astype(int)
    df_feat['is_afternoon'] = ((df_feat['appointment_hour'] >= 12) &
                               (df_feat['appointment_hour'] < 17)).astype(int)
    df_feat['is_evening'] = (df_feat['appointment_hour'] >= 17).astype(int)

    #3. Lead time categories
    df_feat['lead_time_category'] = pd.cut(
        df_feat['lead_time_days'],
        bins=[-1, 1, 3, 7, 14, 100],
        labels=['same_day', 'short', 'medium', 'long', 'very_long'],
        include_lowest=True
    )

    
    #4. Age Groups
    df_feat['age_group'] = pd.cut(df_feat['age'],
                                    bins=[0, 18, 35, 50, 65, 120],
                                    labels=['child', 'young_adult', 'adult', 'senior', 'elderly'])

    #5. Wait time categories (using imputed values)
    df_feat['wait_time_category'] = pd.cut(df_feat['wait_time_minutes'],
                                           bins=[-1, 10, 20, 30, 60, 999],
                                           labels=['excellent', 'good', 'average', 'poor', 'very_poor'])
    
    #6. Risk scores
    df_feat['lead_time_risk'] = np.where(df_feat['lead_time_days'] > 14, 1, 0)
    df_feat['age_risk'] = np.where(df_feat['age'] < 25, 1,
                                   np.where(df_feat['age'] > 65, 1, 0))
    df_feat['wait_time_risk'] = np.where(df_feat['wait_time_minutes'] > 30, 1, 0)
    
    #7. Historical patterns (simulated - in production use real history)
    # Create patient-Level aggregates
    patient_stats = df_feat.groupby('patient_id').agg({
        'is_no_show_0_1': ['mean', 'count'],
        'lead_time_days': 'mean',
        'wait_time_minutes': 'mean'
    }).round(2)
    
    patient_stats.columns = ['patient_no_show_rate', 'patient_visit_count',
                             'patient_avg_lead_time', 'patient_avg_wait_time']
    
    patient_stats = patient_stats.reset_index()

    #Merge back 
    df_feat = df_feat.merge(patient_stats, on='patient_id', how='left')    
    
    # Fill NaNs created by patient-level aggregates
    df_feat['patient_no_show_rate'] = df_feat['patient_no_show_rate'].fillna(0)
    df_feat['patient_visit_count'] = df_feat['patient_visit_count'].fillna(0)
    df_feat['patient_avg_lead_time'] = df_feat['patient_avg_lead_time'].fillna(
        df_feat['patient_avg_lead_time'].median()
    )
    df_feat['patient_avg_wait_time'] = df_feat['patient_avg_wait_time'].fillna(
        df_feat['patient_avg_wait_time'].median()
    )

    #8. Encode categorical variables 
    categorical_cols = ['insurance_type', 'specialty', 'clinic_name', 'city'] 
    
    label_encoders = {}
    for col in categorical_cols: 
        le = LabelEncoder() 
        df_feat[f'{col}_encoded'] = le.fit_transform(df_feat[col].astype(str)) 
        label_encoders[col] = le

    # 9. Interaction features 
    df_feat['morning_elderly'] = df_feat['is_morning'] * (df_feat['age'] > 60).astype(int)
    df_feat['long_lead_public'] = (df_feat['lead_time_risk'] * (df_feat['insurance_type'] == 'Public').astype(int))
    df_feat['high_risk_patient'] = (df_feat['patient_no_show_rate'] > 0.3).astype(int)

    print("\nFeature engineering completed") 
    print(f"Total features created: {df_feat.shape[1] - final_df.shape[1]}")

    from sklearn.impute import SimpleImputer

    # Identify numeric and categorical columns 
    num_cols = df_feat.select_dtypes(include=['int64', 'float64']).columns.tolist() 
    cat_cols = df_feat.select_dtypes(include=['object', 'category']).columns.tolist() 
    
    # Drop numeric columns that are entirely NaN 
    num_cols = [col for col in num_cols if not df_feat[col].isna().all()] 
    
    # Impute numeric columns 
    if len(num_cols) > 0: 
        df_feat[num_cols] = SimpleImputer(strategy='median').fit_transform(df_feat[num_cols])
        
    # Impute categorical columns 
    if len(cat_cols) > 0: 
        df_feat[cat_cols] = SimpleImputer(strategy='most_frequent').fit_transform(df_feat[cat_cols]) 
        
    # 
    return df_feat, label_encoders

    

In [423]:
#=============================================================================
# 8. MODEL PREPARATION
# =============================================================================
def prepare_model_data(df_feat, target_col="is_no_show_0_1"):
    """Select features and split data for modeling"""

    # Define features to use
    feature_cols = [
        # Numerical features
        'lead_time_days',
        'wait_time_minutes',
        'age',
        'appointment_month',
        'appointment_dayofweek',
        'appointment_hour',
        'patient_no_show_rate',
        'patient_visit_count',
        'patient_avg_lead_time',
        'patient_avg_wait_time',

        # Binary features
        'is_monday',
        'is_morning',
        'is_weekend',
        'is_morning',
        'is_afternoon',
        'is_evening',
        'lead_time_risk',
        'age_risk',
        'wait_time_risk',
        'morning_elderly',
        'long_lead_public',
        'high_risk_patient',
        'wait_time_imputed',

        # Encoded categorical features
        'insurance_type_encoded',
        'specialty_encoded',
        'clinic_name_encoded',
        'city_encoded',

        # Interaction features
        'morning_elderly',
        'long_lead_public',
        'high_risk_patient'
    ]

    # Ensure all selected columns exist
    available_features = [c for c in feature_cols if c in df_feat.columns]
    print(f"\nUsing {len(available_features)} features for modeling")

    X = df_feat[available_features]
    y = df_feat[target_col]

    # Check for any remaining missing values 
    missing_in_features = X.isnull().sum().sum()
    if missing_in_features > 0: 
        print(f"Warning: {missing_in_features} missing values found in features.") 
        print("Filling missing values with median for numerical columns.") 
        X = X.fillna(X.median()) 
    
    # Train-test split from sklearn.model_selection import train_test_split 
    X_train, X_test, y_train, y_test = train_test_split( 
        X, 
        y, 
        test_size=0.2, 
        random_state=42, 
        stratify=y 
    ) 
    
    # Scale numerical features from sklearn.preprocessing import StandardScaler 
    scaler = StandardScaler() 
    numerical_cols = ["lead_time_days", "wait_time_minutes", "age", 
                       "appointment_month", "appointment_hour", 
                       "patient_no_show_rate", "patient_visit_count", 
                       "patient_avg_lead_time", "patient_avg_wait_time"] 
    
    # Keep only columns that actually exist in X 
    numerical_cols = [c for c in numerical_cols if c in X.columns]

    # Copy datasets 
    X_train_scaled = X_train.copy() 
    X_test_scaled = X_test.copy() 
    
    # Fit on training data, transform both sets 
    X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols]) 
    X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols]) 
    
    # Print dataset sizes 
    print(f"Training set: {X_train.shape[0]} samples") 
    print(f"Test set: {X_test.shape[0]} samples") 
    
    # Return processed outputs 
    return X_train_scaled, X_test_scaled, y_train, y_test, scaler, available_features

In [449]:
#=============================================================================
# 9. MODEL TRAINING
# =============================================================================

def train_models(X_train, y_train, X_test, y_test):
    """Train multiple models and select the best one"""

    models = {
        "Logistic Regression": LogisticRegression(
            random_state=42,
            max_iter=1000,
            class_weight='balanced'
        ),

        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight='balanced'
        ),

        "XGBoost": xgb.XGBClassifier(
            random_state=42,
            eval_metric='logloss',
            scale_pos_weight=4
        )
    }

    results = {}
    best_model = None
    best_score = 0

    for name, model in models.items():
        # Train model
        model.fit(X_train, y_train)

        # Predictions
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]

        # Metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_pred_proba)

        # Cross-validation
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')

        results[name] = {
            'model': model,
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'roc_auc': roc_auc,
            'cv_mean_accuracy': cv_scores.mean(),
            'cv_std': cv_scores.std()
        }
        
        print(f"\n{'='*50}")
        print(f"{name} Results:")
        print(f"{'='*50}")

        print(f"Accuracy:     {accuracy:.4f}  (Target: 0.75)")
        print(f"Precision:    {precision:.4f}")
        print(f"Recall:       {recall:.4f}")
        print(f"F1-Score:     {f1:.4f}")
        print(f"ROC-AUC:      {roc_auc:.4f}")
        print(f"CV Score:     {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

        # Check if this is the best model
        if accuracy > best_score:
            best_score = accuracy
            best_model = model
            best_model_name = name

        print(f"\n{'='*50}")
        print(f"BEST MODEL: {best_model_name} with Accuracy: {best_score:.4f}")
        print(f"{'='*50}")

        return results, best_model, best_model_name




In [450]:
#=============================================================================
# 9. ANALYZE FEATURE IMPORTANCE
# =============================================================================
def analyze_feature_importance(model, feature_names, model_name):
    """Analyze and plot feature importance"""

    # Check if model supports feature_importances_
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        indices = np.argsort(importances)[::-1]

        # Plot top 10 features
        plt.figure(figsize=(12, 6))
        plt.title(f'Feature Importance - {model_name}')
        plt.bar(range(10), importances[indices][:10])
        plt.xticks(range(10),
                   [feature_names[i] for i in indices[:10]],
                   rotation=45)
        plt.tight_layout()
        plt.savefig('feature_importance.png', dpi=100)
        plt.show()

        # Print top 10 features
        print("\nTop 10 Important Features:")
        for i in range(10):
            print(f"{i+1}. {feature_names[indices[i]]}: {importances[indices[i]]:.4f}")

    else:
        print(f"Model '{model_name}' does not provide feature_importances_.")


In [451]:
def save_model_artifacts(model, scaler, label_encoders, feature_names, model_name, imputation_stats=None):
    """Save model and all preprocessing artifacts"""

    artifacts = {
        "model": model,
        "scaler": scaler,
        "label_encoders": label_encoders,
        "feature_names": feature_names,
        "model_name": model_name,
        "version": "1.0",
        "created_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "imputation_stats": imputation_stats or {}
    }

    # Save using joblib
    joblib.dump(artifacts, "no_show_model.pkl")
    print("Model artifacts saved successfully.")

    # Also save individual components
    joblib.dump(model, "model_component.pkl")
    joblib.dump(scaler, "scaler_component.pkl")

    print("\n" + "="*50)
    print("MODEL SAVED SUCCESSFULLY")
    print("="*50)

    print(f"Model: {model_name}")
    print(f"Features: {len(feature_names)}")

    print("Files created:")
    print(" - no_show_model.pkl (complete pipeline)")
    print(" - model_component.pkl")
    print(" - scaler_component.pkl")
    print("="*50)


In [452]:
def main(df):
    """Main execution function"""
    
    print("\n" + "=" * 60)
    print("HEALTHPULSE NO-SHOW PREDICTION MODEL")
    print("=" * 60)

    print("\n[2] Missing values already handled earlier...")

    # 1. Feature engineering
    print("\n[3] Engineering features...")
    df_feat, label_encoders = engineer_features(df)

    # 2. Prepare for modeling
    print("\n[4] Preparing data for modeling...")
    X_train, X_test, y_train, y_test, scaler, feature_names = prepare_model_data(df_feat)

    # 3. Train models
    print("\n[5] Training models...")
    results, best_model, best_model_name = train_models(X_train, y_train, X_test, y_test)

    # 4. Feature importance
    print("\n[6] Analyzing feature importance...")
    analyze_feature_importance(best_model, feature_names, best_model_name)

    # 5. Save model
    print("\n[7] Saving model artifacts...")
    save_model_artifacts(
        best_model,
        scaler,
        label_encoders,
        feature_names,
        best_model_name
    )

    print("\n" + "=" * 60)
    print("PIPELINE COMPLETED SUCCESSFULLY")
    print("=" * 60)

    return results, best_model


# Run the main function
if __name__ == "__main__":
    results, model = main(final_df)





HEALTHPULSE NO-SHOW PREDICTION MODEL

[2] Missing values already handled earlier...

[3] Engineering features...

Feature engineering completed
Total features created: 23

[4] Preparing data for modeling...

Using 29 features for modeling
Training set: 96000 samples
Test set: 24000 samples

[5] Training models...

Logistic Regression Results:
Accuracy:     0.6984  (Target: 0.75)
Precision:    0.3510
Recall:       0.8192
F1-Score:     0.4914
ROC-AUC:      0.8045
CV Score:     0.6956 (+/- 0.0021)

BEST MODEL: Logistic Regression with Accuracy: 0.6984

[6] Analyzing feature importance...
Model 'Logistic Regression' does not provide feature_importances_.

[7] Saving model artifacts...
Model artifacts saved successfully.

MODEL SAVED SUCCESSFULLY
Model: Logistic Regression
Features: 29
Files created:
 - no_show_model.pkl (complete pipeline)
 - model_component.pkl
 - scaler_component.pkl

PIPELINE COMPLETED SUCCESSFULLY
